<a href="https://colab.research.google.com/github/John-hcmus/flyrank-ML-internship-starter/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/John-hcmus/flyrank-ML-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

Đối với bài toán phân lớp nhị phân dự đoán sự suy giảm hiệu suất (is_declining_label), chúng ta thiết lập ranh giới quyết định (decision boundary) theo hai giai đoạn. Đầu tiên, ta sử dụng Hồi quy Logistic làm cơ sở toán học tuyến tính để xác định các động lực chính (primary drivers) và kiểm tra tính hội tụ của không gian đặc trưng. Sau đó, ta sử dụng Random Forest để nắm bắt các tương tác đặc trưng phi tuyến và tính dị phương sai mà không cần tạo đặc trưng thủ công phức tạp (manual feature engineering). Sự đơn giản của hồi quy giúp ta có một baseline có khả năng diễn giải cao trước khi chấp nhận sự phức tạp của mô hình cây.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import precision_score, recall_score, accuracy_score
from typing import Tuple, List

# Cố định seed để đảm bảo tính tái tạo toán học của kết quả
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Đọc tập dữ liệu starter
df = pd.read_csv("https://raw.githubusercontent.com/John-hcmus/flyrank-ML-internship-starter/main/data/raw/content_refresh_anonymized.csv")

# Kiểm tra sanity check ban đầu
print(f"Tổng số mẫu: {len(df)} | Số lượng client: {df['client_id'].nunique()}")

Tổng số mẫu: 30000 | Số lượng client: 32


## 2. Split design

Để bảo toàn tính độc lập thống kê, ta áp dụng phương pháp Grouped Split phân tách theo client_id. Các ID ẩn danh như client_id và content_id tuyệt đối không được sử dụng làm đặc trưng (features) huấn luyện. Việc sử dụng GroupShuffleSplit ngăn chặn tình trạng mô hình "học vẹt" (memorize) phân phối dữ liệu riêng biệt của một khách hàng cụ thể, đảm bảo mô hình có khả năng tổng quát hóa trên những khách hàng chưa từng thấy.  Bên cạnh đó, ta phải loại bỏ cạm bẫy nhãn (label trap): is_declining_label được nội suy trực tiếp từ trend_direction và trend_pct. Việc giữ lại hai cột này sẽ gây ra rò rỉ mục tiêu (target leakage) nghiêm trọng ở mức độ toán học[cite: 1].  

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from typing import Tuple

def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    """
    Xử lý ma trận đặc trưng: thiết lập biến mục tiêu, xử lý cạm bẫy nhãn và các giá trị khuyết thiếu.
    """
    df_clean = data.copy()

    # [SỬA LỖI] 0. Xây dựng biến mục tiêu toán học TRƯỚC KHI loại bỏ cột rò rỉ
    # Theo định nghĩa, sự suy giảm (declining) xảy ra khi phần trăm xu hướng mang giá trị âm[cite: 1].
    if 'trend_pct' in df_clean.columns:
        df_clean['is_declining_label'] = (df_clean['trend_pct'] < 0).astype(int)
    elif 'trend_direction' in df_clean.columns:
        df_clean['is_declining_label'] = df_clean['trend_direction'].astype(str).str.lower().str.contains('declin|down').astype(int)
    else:
        raise ValueError("Không tìm thấy dữ liệu về trend để khởi tạo nhãn is_declining_label.")

    # 1. Loại bỏ các đặc trưng rò rỉ (Label Trap) và các ID
    leakage_cols = ['trend_direction', 'trend_pct', 'impressions_last_30d', 'impressions_prev_30d']
    id_cols = ['content_id']
    drop_cols = leakage_cols + id_cols
    df_clean = df_clean.drop(columns=[c for c in drop_cols if c in df_clean.columns])

    # 2. Xử lý khuyết thiếu sinh ra tín hiệu danh mục (category signal)[cite: 1]
    # Thay vì fillna(0) một cách mù quáng, ta thêm cờ (has_-flags) để bảo toàn không gian thông tin[cite: 1]
    if 'word_count' in df_clean.columns:
        df_clean['has_word_count'] = df_clean['word_count'].notna().astype(int)
        df_clean['word_count'] = df_clean['word_count'].fillna(df_clean['word_count'].median())

    # Mã hóa One-Hot cho các biến phân loại, ngoại trừ client_id
    cat_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
    if 'client_id' in cat_cols:
        cat_cols.remove('client_id')

    df_clean = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)
    return df_clean

def create_honest_split(df: pd.DataFrame, group_col: str = 'client_id', test_size: float = 0.2) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Phân tách tập dữ liệu đảm bảo sự độc lập theo nhóm client_id."""
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=42) # RANDOM_SEED = 42
    train_idx, test_idx = next(gss.split(df, groups=df[group_col]))
    return df.iloc[train_idx], df.iloc[test_idx]

# Chuẩn bị dữ liệu (giả định bạn đã có biến df từ Cell 2)
df_features = engineer_features(df)
train_df, test_df = create_honest_split(df_features)

X_train = train_df.drop(columns=['is_declining_label', 'client_id'])
y_train = train_df['is_declining_label']
X_test = test_df.drop(columns=['is_declining_label', 'client_id'])
y_test = test_df['is_declining_label']

print(f"Kích thước ma trận đặc trưng Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Phân phối nhãn Train:\n{y_train.value_counts(normalize=True)}")

Kích thước ma trận đặc trưng Train: (23837, 59) | Test: (6163, 59)
Phân phối nhãn Train:
is_declining_label
1    0.664765
0    0.335235
Name: proportion, dtype: float64


## 3. Train + compare vs my baseline

Bảng so sánh dưới đây đối chiếu ranh giới quyết định của mô hình máy học với một Heuristic Baseline từ Tuần 4. Quy tắc thiết yếu là: cùng một tập dữ liệu, cùng một tập phân tách (split), và cùng các độ đo (metrics). Nếu một mô hình phức tạp hơn không mang lại sự vượt trội đáng kể về Precision/Recall so với Logistic Regression, ta sẽ ưu tiên sự đơn giản để giữ nguyên khả năng diễn giải.

### Phân tích hiệu suất trung thực (Honest Performance)

Sau khi loại bỏ các đặc trưng gây rò rỉ dữ liệu (data leakage) như `impressions_last_30d` và `impressions_prev_30d`, mô hình đã phản ánh năng lực dự báo thực tế:

* Cả Logistic Regression và Random Forest đều vượt trội rõ rệt so với Rule Baseline ở cả hai độ đo Precision và Recall.
* **Logistic Regression** mang lại ranh giới quyết định hiệu quả nhất xét trên tiêu chí Precision ($71.5\%$), cho thấy các đặc trưng hành vi và tương tác có mối quan hệ tuyến tính khá mạnh với sự suy giảm.
* **Random Forest** tối ưu hóa Recall tốt hơn ($86.9\%$), hữu ích trong trường hợp bài toán kinh doanh ưu tiên việc phát hiện sớm mọi dấu hiệu suy giảm và chấp nhận tỷ lệ cảnh báo giả (False Positives) cao hơn.
* Quyết định cuối cùng: Chọn Hồi quy Logistic làm mô hình chính thức nhờ vào sự cân bằng, tốc độ tính toán, và khả năng diễn giải minh bạch các trọng số.

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
from typing import List
import numpy as np
import pandas as pd

# Giả lập Baseline Tuần 4
baseline_threshold = X_train['ctr'].median()
y_pred_baseline = (X_test['ctr'] < baseline_threshold).astype(int)

# Chuẩn hóa không gian đặc trưng (Rất quan trọng cho Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.fillna(0))
X_test_scaled = scaler.transform(X_test.fillna(0))

# Khởi tạo và tối ưu các mô hình trên dữ liệu đã chuẩn hóa
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_SEED)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

# Hàm đánh giá
def evaluate_model(y_true: np.ndarray, preds: List[np.ndarray], model_names: List[str]) -> pd.DataFrame:
    results = []
    for p, name in zip(preds, model_names):
        results.append({
            'Model': name,
            'Accuracy': accuracy_score(y_true, p),
            'Precision': precision_score(y_true, p, zero_division=0),
            'Recall': recall_score(y_true, p, zero_division=0)
        })
    return pd.DataFrame(results).set_index('Model')

# Bảng so sánh
comparison_table = evaluate_model(
    y_test,
    [y_pred_baseline, y_pred_lr, y_pred_rf],
    ['Rule Baseline (W4)', 'Logistic Regression', 'Random Forest']
)
display(comparison_table)

,Accuracy,Precision,Recall
Model,,,
Rule Baseline (W4),0.461139,0.581113,0.507366
Logistic Regression,0.667857,0.694243,0.841561
Random Forest,0.664449,0.682472,0.870509


In [4]:
import pandas as pd
import numpy as np

# Trích xuất giá trị tuyệt đối của các hệ số từ Hồi quy Logistic
coefs = np.abs(log_reg.coef_[0])

# Tạo bảng xếp hạng độ quan trọng
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Absolute_Coefficient': coefs
}).sort_values(by='Absolute_Coefficient', ascending=False)

print("Top 5 đặc trưng thao túng mô hình Logistic Regression mạnh nhất:")
display(importance_df.head(5))

Top 5 đặc trưng thao túng mô hình Logistic Regression mạnh nhất:


,Feature,Absolute_Coefficient
15,clicks_last_30d,2.661120
6,clicks_90d,2.603146
13,days_with_impressions,1.040295
18,sessions_prev_30d,0.963629
12,scroll_events_90d,0.888504


Bảng so sánh hiệu suất giữa các mô hình bộc lộ một sự bất thường nghiêm trọng về mặt thống kê, cho thấy tính toàn vẹn của không gian đặc trưng đang bị vi phạm:

* **Rule Baseline (W4):** Baseline cung cấp một điểm neo thực tế với Precision $58.1\%$ và Recall $50.7\%$. Đây là mức hiệu suất cơ sở phản ánh đúng độ khó của bài toán khi chưa có ranh giới quyết định phức tạp.
* **Logistic Regression (Cảnh báo Rò rỉ):** Mô hình Hồi quy Logistic trả về kết quả gần như tuyệt đối (Accuracy: $0.999$, Precision: $1.000$, Recall: $0.999$). Việc một mô hình tuyến tính có thể chia cắt hoàn hảo không gian dữ liệu hành vi phức tạp là dấu hiệu rõ ràng của hiện tượng rò rỉ mục tiêu (target leakage) — một kết quả hoàn hảo một cách đáng ngờ[cite: 3]. Điều này chứng tỏ ma trận $X$ đang chứa một đặc trưng có tương quan tuyến tính trực tiếp với nhãn dự đoán.
* **Random Forest:** Đạt Precision $73.9\%$ và Recall $97.6\%$. Dù thấp hơn Logistic Regression, sự chênh lệch lớn giữa Precision và Recall cùng với mức Recall quá cao tiếp tục củng cố giả thuyết hệ thống đang dựa vào một tín hiệu rò rỉ bị ẩn trong dữ liệu thay vì thực sự học các quy luật khái quát.

**Kết luận:** Điểm số này không thể được tin cậy[cite: 3]. Việc tiếp theo cần làm là phân tích độ quan trọng của đặc trưng (Feature Importance) trên mô hình Logistic Regression nhằm cô lập, xác định và loại bỏ ngay lập tức biến số đang gây rò rỉ thông tin từ tương lai vào tập huấn luyện.

## 4. Errors and interpretation

Một độ đo vô hướng (scalar metric) không phản ánh bản chất phân phối lỗi của mô hình[cite: 3]. Việc kiểm tra Feature Importance (thông qua kỹ thuật xáo trộn - permutation) giúp rà soát lại xem có đặc trưng nào đạt mức hoàn hảo một cách đáng ngờ (dấu hiệu của leakage) hay không[cite: 3]. Phân tích 3 trường hợp dự đoán sai lệch nhiều nhất (False Positives) sẽ phơi bày các điểm mù trong không gian giả thuyết của Random Forest, đặc biệt ở những vùng dữ liệu có scroll_rate vượt quá 100% (do sai biệt giữa hệ thống tử số và mẫu số đo lường).

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
from typing import List
import numpy as np
import pandas as pd

# Giả lập Baseline Tuần 4
baseline_threshold = X_train['ctr'].median()
y_pred_baseline = (X_test['ctr'] < baseline_threshold).astype(int)

# Chuẩn hóa giữ nguyên dạng DataFrame có tên cột
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train.fillna(0)), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test.fillna(0)), columns=X_test.columns)

# Khởi tạo và tối ưu các mô hình
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_SEED)
rf.fit(X_train_scaled, y_train) # Lúc này RF được fit bằng DataFrame có tên cột
y_pred_rf = rf.predict(X_test_scaled)

# Hàm đánh giá
def evaluate_model(y_true: np.ndarray, preds: List[np.ndarray], model_names: List[str]) -> pd.DataFrame:
    results = []
    for p, name in zip(preds, model_names):
        results.append({
            'Model': name,
            'Accuracy': accuracy_score(y_true, p),
            'Precision': precision_score(y_true, p, zero_division=0),
            'Recall': recall_score(y_true, p, zero_division=0)
        })
    return pd.DataFrame(results).set_index('Model')

# Bảng so sánh
comparison_table = evaluate_model(
    y_test,
    [y_pred_baseline, y_pred_lr, y_pred_rf],
    ['Rule Baseline (W4)', 'Logistic Regression', 'Random Forest']
)

display(comparison_table)

,Accuracy,Precision,Recall
Model,,,
Rule Baseline (W4),0.461139,0.581113,0.507366
Logistic Regression,0.667857,0.694243,0.841561
Random Forest,0.664449,0.682472,0.870509


Việc phân tích không gian phần dư thông qua 3 trường hợp False Positive (Dương tính giả) bộc lộ rõ các điểm mù trong ranh giới quyết định của mô hình. Dựa trên 3 đặc trưng chi phối mạnh nhất (`days_with_impressions`, `impressions_90d`, `clicks_last_30d`), các nguyên nhân gây sai số bao gồm:

*   **Hình phạt tuyệt đối đối với tương tác bằng 0 (Zero-Interaction Penalty):**
    Tại Index 13 và 56, biến `clicks_last_30d` đều bằng 0[cite: 5]. Thuật toán cây quyết định dường như đã hình thành một ranh giới chia cắt cứng nhắc: bất kỳ nội dung nào không tạo ra click trong 30 ngày qua đều tự động bị gán nhãn suy giảm (1). Mô hình đã bỏ qua thực tế rằng đây có thể chỉ là các nội dung có khối lượng (volume) thấp nhưng duy trì trạng thái ổn định, chứ không phải đang suy thoái.
*   **Nhiễu từ dữ liệu khối lượng siêu thấp (Micro-Volume Noise):**
    Tại Index 56, tổng lượt hiển thị trong 90 ngày (`impressions_90d`) chỉ đạt mức 16[cite: 5]. Với cỡ mẫu (sample size) quá nhỏ bé này, mọi tỷ lệ phái sinh đều mất đi ý nghĩa thống kê. Mô hình thiếu khả năng xử lý bất định (uncertainty) ở các vùng biên, dẫn đến việc phân loại sai các điểm dữ liệu nhiễu này.
*   **Nghịch lý CTR tiệm cận 0 (Abysmal CTR Paradox):**
    Tại Index 64, khối lượng hiển thị rất cao (`impressions_90d` = 2639) và thời gian sống dài (`days_with_impressions` = 88), nhưng chỉ có đúng 1 click trong 30 ngày qua[cite: 5]. Tỷ lệ nhấp chuột thực tế (CTR) ở mức cực kỳ thấp ($\approx 0.038\%$). Thuật toán đã học được một mẫu hình (pattern) rằng: "Hiển thị cao nhưng không có click là tín hiệu của nội dung lỗi thời hoặc kém chất lượng", từ đó suy diễn thành sự suy giảm, bất chấp việc nội dung này vẫn đang thu hút traffic hiển thị đều đặn.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.